In [ ]:
#|default_exp coinbase

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

## Coinbase Refactored

This notebook refactors the original `coinbase.ipynb` to use the `DataSource` abstract base class and `polars` instead of `pandas`.

In [ ]:
#| export

from token_data.datasource import DataSource
from pathlib import Path
from typing import List
import polars as pl
import requests
import logging
from datetime import datetime, timedelta

In [ ]:
#| export

class CoinbaseDataSource(DataSource):
    """A data source for Coinbase."""

    def __init__(self, data_folder: Path):
        super().__init__(data_folder)
        self.api_url = "https://api.exchange.coinbase.com"
        self.time_interval_map = {
            '1m': 60,
            '5m': 300,
            '15m': 900,
            '1h': 3600,
            '4h': 14400,
            '1d': 86400
        }

    def get_tokens(self) -> List[str]:
        """Returns a list of available tokens."""
        url = f"{self.api_url}/products"
        response = requests.get(url)
        response.raise_for_status()
        tokens = response.json()
        return [token['id'] for token in tokens]

    def get_spot_prices(self, token: str, start_date: str, end_date: str, time_interval: str = '1h') -> pl.DataFrame:
        """Returns a DataFrame of prices for a given token."""
        granularity = self.time_interval_map.get(time_interval)
        if granularity is None:
            raise ValueError(f"Unsupported time interval: {time_interval}")
        url = f"{self.api_url}/products/{token}/candles"
        params = {
            'start': start_date,
            'end': end_date,
            'granularity': granularity
        }
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        if not data:
            return pl.DataFrame()
        
        df = pl.DataFrame(data, schema=[
            ('time', pl.Int64),
            ('low', pl.Float64),
            ('high', pl.Float64),
            ('open', pl.Float64),
            ('close', pl.Float64),
            ('volume', pl.Float64)
        ])
        df = df.with_columns([
            pl.from_epoch(pl.col('time'), time_unit='s').alias('datetime'),
            pl.lit(token).alias('token')
        ])
        return df.select(['datetime', 'open', 'high', 'low', 'close', 'volume', 'token'])
    
    def get_perp_prices(self, token: str, start_date: str, end_date: str, time_interval: str) -> pl.DataFrame:
        raise NotImplementedError("Coinbase does not support perpetual futures.")
        
    def get_funding_rates(self, token: str, start_date: str, end_date: str) -> pl.DataFrame:
        raise NotImplementedError("Coinbase does not support funding rates.")


### Tests

In [ ]:
import unittest
from pathlib import Path
import shutil

class TestCoinbaseDataSource(unittest.TestCase):
    
    def setUp(self):
        self.data_folder = Path('./test_data')
        self.data_folder.mkdir(exist_ok=True)
        self.coinbase = CoinbaseDataSource(self.data_folder)

    def tearDown(self):
        shutil.rmtree(self.data_folder)

    def test_get_tokens(self):
        tokens = self.coinbase.get_tokens()
        self.assertIsInstance(tokens, list)
        self.assertGreater(len(tokens), 0)
        self.assertIn('BTC-USD', tokens)

    def test_get_spot_prices(self):
        end_date = datetime.now()
        start_date = end_date - timedelta(days=1)
        df = self.coinbase.get_spot_prices('BTC-USD', start_date.isoformat(), end_date.isoformat(), '1h')
        self.assertIsInstance(df, pl.DataFrame)
        self.assertGreater(len(df), 0)
        self.assertEqual(df.columns, ['datetime', 'open', 'high', 'low', 'close', 'volume', 'token'])

    def test_get_perp_prices(self):
        with self.assertRaises(NotImplementedError):
            self.coinbase.get_perp_prices('BTC-USD', '', '', '')
            
    def test_get_funding_rates(self):
        with self.assertRaises(NotImplementedError):
            self.coinbase.get_funding_rates('BTC-USD', '', '')

    def test_save_and_read_data(self):
        df = pl.DataFrame({
            'datetime': [datetime.now()],
            'open': [1.0],
            'high': [2.0],
            'low': [0.5],
            'close': [1.5],
            'volume': [100.0],
            'token': ['BTC-USD']
        })
        self.coinbase.save_data(df, 'BTC-USD', 'parquet')
        read_df = self.coinbase.read_data('BTC-USD', 'parquet')
        self.assertTrue(df.equals(read_df))
        
        self.coinbase.save_data(df, 'BTC-USD', 'csv')
        read_df = self.coinbase.read_data('BTC-USD', 'csv')
        self.assertTrue(df.equals(read_df))

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)